 # pyMBE Tutorial: the Python-based Molecule Builder for ESPResSo.





The objective of pyMBE is to facilitate the set-up of complex molecules into the Molecular Dynamics software [espressomd.org](ESPResSo). pyMBE is specially well-suitedto facilitate setting up constant pH and grand-reaction ensemble simulations in ESPResSo. 

## Table of contents:
* [Introduction](#introduction)
* [How to create particles](#particles)
* [How to create simple polymers](#simple_polymers)
* [How to create complex polymers](#complex_polymers)
* [How to create di-block copolymers](#diblock_copolymers)
* [Practice by creating branched polyampholytes](#branched_polyampholytes)
* [How to create peptides](#peptides)

## Introduction <a class="anchor" id="introduction"></a>

Let us get started by importing pyMBE library and other important libraries for this tutorial, such as ESPResSo.

In [1]:
# Import pyMBE and  ESPResSo
import pyMBE
import espressomd
from espressomd import interactions

Creating an instance of pyMBE we can make our code shorter and more readable. 

In [2]:
import pyMBE
pmb = pyMBE.pymbe_library(seed=42)

When pyMBE is inicialized, a default system of reduced units is defined. 

* Unit_length = 0.355 nm.
* Unit_charge = 1 elementary charge.
* Temperature = 298.15 K.

The active set of reduced units can be consulted using:

In [3]:
reduced_unit_set = pmb.get_reduced_units()
print(reduced_unit_set)

Current set of reduced units:
0.355 nanometer = 1 reduced_length
4.1164e-21 joule = 1 reduced_energy
1.6022e-19 coulomb = 1 reduced_charge
Temperature: 298.15 kelvin


In [4]:
print(reduced_unit_set)

Current set of reduced units:
0.355 nanometer = 1 reduced_length
4.1164e-21 joule = 1 reduced_energy
1.6022e-19 coulomb = 1 reduced_charge
Temperature: 298.15 kelvin


This default definition of reduced units can be changed at the convience of the user using the following command:

In [5]:
pmb.set_reduced_units(unit_length = 0.5*pmb.units.nm,  
                      unit_charge = 5*pmb.units.e)
                      #, temperature=300*pmb.units.K)

In [6]:
reduced_unit_set = pmb.get_reduced_units()
print(reduced_unit_set)

Current set of reduced units:
0.5 nanometer = 1 reduced_length
4.1164e-21 joule = 1 reduced_energy
8.0109e-19 coulomb = 1 reduced_charge
Temperature: 298.15 kelvin


NOTE: All input variables will be given to ESPResSo using these reduced units, since it is a convenient choice for the simulation setup. Internally, pyMBE uses Pint library to deal with unit transformations, which in turn should be  used by the user to define its own variables.

Let us now create an instance of the ESPResSo system where we will place our molecules (a square simulation box with length = `box_l`).

In [7]:
Box_L = 7.5*pmb.units.nm
box_l=[Box_L.to('reduced_length').magnitude]*3

espresso_system = espressomd.System(box_l = box_l)

print('The side of the simulation box is ', Box_L, '=' ,Box_L.to('reduced_length'))
print(type(espresso_system),"type")

The side of the simulation box is  7.5 nanometer = 14.999999999999998 reduced_length
<class 'espressomd.system.System'> type


## How to create particles <a class="anchor" id="particles"></a>

Particles are the smaller objects in the simulation box, which can represent small ions or other small chemical species. In turn, particles can also be used as building blocks for larger molecules or polymers, where they can represent one monomeric unit or part of it. Particle objects are used in pyMBE as input for several of its funcionalities, including to create larger molecules and peptides. The basic properties of a particle are:

In [9]:
cation_name = 'Na'
pmb.define_particle(name = cation_name, 
                    z = 0, 
                    sigma = 0.35*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pyMBE stores the properties of each different particle type or particle "template" on its internal database. To check all particle templates defined in the pyMBE database, one can query the pyMBE database manager:

In [10]:
pmb.get_templates_df(pmb_type = 'particle')

,pmb_type,name,sigma,epsilon,cutoff,offset,initial_state
0,particle,Na,0.35 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,Na


One can use pyMBE to create any number of the defined particles into the ESPResSo system.

In [11]:
cation_name
pmb.db._has_template(name=cation_name, pmb_type="particle")

True

In [12]:
N_cations = 20
pmb.create_particle(name = cation_name,
                    number_of_particles = N_cations,
                    box_l=box_l)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

pyMBE keeps track of each instance of a particle template created into the ESPResSo system. To check all particle instances created, one can query the pyMBE database manager:

In [13]:
pmb.get_instances_df(pmb_type = 'particle')

,pmb_type,name,particle_id,position,fix,initial_state,residue_id,molecule_id,assembly_id
0,particle,Na,0,"[11.609340728339449, 6.583176596280784, 12.878...",None,Na,<NA>,<NA>,<NA>
1,particle,Na,1,"[10.460520435890457, 1.4126602183147428, 14.63...",None,Na,<NA>,<NA>,<NA>
2,particle,Na,2,"[11.417095529855294, 11.790964579154306, 1.921...",None,Na,<NA>,<NA>,<NA>
3,particle,Na,3,"[6.755789068433506, 5.561970363488718, 13.9014...",None,Na,<NA>,<NA>,<NA>
4,particle,Na,4,"[9.657976801209967, 12.341424199062448, 6.6512...",None,Na,<NA>,<NA>,<NA>
5,particle,Na,5,"[3.408580826771653, 8.318771805237521, 0.95725...",None,Na,<NA>,<NA>,<NA>
6,particle,Na,6,"[12.41446757988873, 9.474965986830972, 11.3713...",None,Na,<NA>,<NA>,<NA>
7,particle,Na,7,"[5.3178895219480244, 14.560470365923548, 13.39...",None,Na,<NA>,<NA>,<NA>
8,particle,Na,8,"[11.675752456106427, 2.919580617779513, 7.0008...",None,Na,<NA>,<NA>,<NA>
9,particle,Na,9,"[0.6570564868084315, 2.3143423810132173, 10.24...",None,Na,<NA>,<NA>,<NA>


In [14]:
pmb.set_simulation_engine(espresso_system)
pmb.add_instances_to_engine()

Let us see the particles that we have created by visualizing our ESPResSo system.

In [8]:
from PIL import Image
def create_snapshot_of_espresso_system(espresso_system, filename):
    """
    Uses espresso visualizer for creating a snapshot of the current state of the espresso_system

    Args:
        espresso_system(`espressomd.system.System`): Instance of a system object from the espressomd library.
        filename(`str`): Name of the ouput file for the snapshot
    """ 
    from espressomd import visualization
    visualizer = visualization.openGLLive(
            espresso_system, bond_type_radius=[0.3], particle_coloring='type', draw_axis=False, background_color=[1, 1, 1],
    particle_type_colors=[[1.02,0.51,0], # Brown
                        [1,1,1],  # Grey
                        [2.55,0,0], # Red
                        [0,0,2.05],  # Blue
                        [0,0,2.05],  # Blue
                        [2.55,0,0], # Red
                        [2.05,1.02,0]]) # Orange
    visualizer.screenshot(filename)
    return

In [17]:
# Only necessary to produce the pictures used in this tutorial
from PIL import Image
def create_snapshot_of_espresso_system(espresso_system, filename):
    """
    Uses espresso visualizer for creating a snapshot of the current state of the espresso_system

    Args:
        espresso_system(`espressomd.system.System`): Instance of a system object from the espressomd library.
        filename(`str`): Name of the ouput file for the snapshot
    """ 
    from espressomd import visualization
    visualizer = visualization.openGLLive(
            espresso_system, bond_type_radius=[0.3], particle_coloring='type', draw_axis=False, background_color=[1, 1, 1],
    particle_type_colors=[[1.02,0.51,0], # Brown
                        [1,1,1],  # Grey
                        [2.55,0,0], # Red
                        [0,0,2.05],  # Blue
                        [0,0,2.05],  # Blue
                        [2.55,0,0], # Red
                        [2.05,1.02,0]]) # Orange
    visualizer.screenshot(filename)
    return
picture_name = 'cation_system.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                               filename = picture_name)
img = Image.open(picture_name)
img.show()

To delete any instance of a particle created with pyMBE from the ESPResSo system we can use the following command:

In [18]:
# First search for the ids of the particles to delete
particle_id_map = pmb.get_particle_id_map(object_name=cation_name)
# This will delete all particles that we created before
for pid in particle_id_map["all"]:
    pmb.delete_instances_in_system(instance_id=pid, 
                                   pmb_type="particle",
                                  espresso_system = espresso_system)

If we query again the pyMBE database for particle instances in the ESPResSo system, we can observe that all particles have been deleted

In [19]:
pmb.get_instances_df(pmb_type = 'particle')

""



## How to create simple polymers <a class="anchor" id="simple_polymers"></a>

pyMBE can be used to easily construct coarse-grained models of simple polymers. Let us consider a coarse grained model for polydehydroalanaline (PDha) (figure below) in which its monomeric unit can be represented by three beads, as depicted in the schematics below: a backbone bead (grey), a bead for the carboxylic acid group (red) and a bead for the amino group (blue).

<img src="../figs/PDha.png" width=150 height=150 />


To set up such polymer with pyMBE first one has to define templates for the different particles in the monomer.

In [9]:
PDha_backbone_bead = 'BB-PDha'
PDha_carboxyl_bead = 'COOH-PDha'
PDha_amine_bead = 'NH3-PDha'

pmb.define_particle(name = PDha_backbone_bead, 
                    z = 0, 
                    sigma = 0.4*pmb.units.nm,
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDha_carboxyl_bead, 
                    z = 0, 
                    sigma = 0.5*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDha_amine_bead, 
                    z = 0, 
                    sigma = 0.3*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))


In [10]:
pmb.get_templates_df(pmb_type = 'particle')

,pmb_type,name,sigma,epsilon,cutoff,offset,initial_state
0,particle,BB-PDha,0.4 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,BB-PDha
1,particle,COOH-PDha,0.5 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,COOH-PDha
2,particle,NH3-PDha,0.3 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,NH3-PDha


Then, one defines templates for the polymer residues. A residue is composed by a `central_bead` where one or various `side_chains` are attached. Each side chain can contain one particle or other residues. 

In [11]:
PDha_residue = 'PDha_mon'

pmb.define_residue(name = PDha_residue, 
                   central_bead = PDha_backbone_bead,
                   side_chains = [PDha_carboxyl_bead, PDha_amine_bead])


We can check the residue templates we defined in the pyMBE database:

In [12]:
pmb.get_templates_df(pmb_type='residue')

,pmb_type,name,central_bead,side_chains
0,residue,PDha_mon,BB-PDha,"[COOH-PDha, NH3-PDha]"


Once done, one has to define templates for bonds between each pair of different particle templates in the polymer. For simplicity, in this tutorial we assume that all bonds are equal and we set-up all bonds using a harmonic potential with the following arbitrary parameters.

In [13]:
bond_type = 'harmonic'
generic_bond_lenght=0.4 * pmb.units.nm
generic_harmonic_constant = 400 * pmb.units('reduced_energy / reduced_length**2')

harmonic_bond = {'r_0'    : generic_bond_lenght,
                 'k'      : generic_harmonic_constant}

pmb.define_bond(bond_type = bond_type,
                bond_parameters = harmonic_bond,
                particle_pairs = [[PDha_backbone_bead, PDha_backbone_bead],
                                  [PDha_backbone_bead, PDha_carboxyl_bead],
                                  [PDha_backbone_bead, PDha_amine_bead]])

All bond templates we defined are stored in the pyMBE database:

In [14]:
pmb.get_templates_df(pmb_type = 'bond')

,pmb_type,name,bond_type,particle_name1,particle_name2,parameters
0,bond,BB-PDha-BB-PDha,harmonic,BB-PDha,BB-PDha,"{'r_0': 0.4 nanometer, 'k': 41108.126593737354..."
1,bond,BB-PDha-COOH-PDha,harmonic,BB-PDha,COOH-PDha,"{'r_0': 0.4 nanometer, 'k': 41108.126593737354..."
2,bond,BB-PDha-NH3-PDha,harmonic,BB-PDha,NH3-PDha,"{'r_0': 0.4 nanometer, 'k': 41108.126593737354..."


NOTE: Currently, only harmonic and FENE bonds are supported.

Finally, one defines a molecule template. A molecule template is defined by a linear sequence of residues, `residue_list`. For instance a decamer should be created as follows:

In [15]:
PDha_polymer = 'PDha'
N_monomers = 2

pmb.define_molecule(name = PDha_polymer,
                    residue_list = [PDha_residue]*N_monomers)

All defined molecule templates can be consulted in the pyMBE database

In [16]:
pmb.get_templates_df(pmb_type = 'molecule')

,pmb_type,name,residue_list
0,molecule,PDha,"[PDha_mon, PDha_mon]"


After defining a template for the polymer, we are ready to create one PdHa polymer in the center of the simulation box.

In [17]:
N_polymers = 1

molecule_ids = pmb.create_molecule(name = PDha_polymer, 
                                    number_of_molecules = N_polymers,
                                    box_l = box_l, 
                                    list_of_first_residue_positions = [[Box_L.to('reduced_length').magnitude/2]*3]) 

All instances of particles, residues, bonds and molecules created into ESPResSo are bookkept in the pyMBE database:

In [18]:
# Check particle instances
pmb.get_instances_df(pmb_type = 'particle')



,pmb_type,name,particle_id,position,fix,initial_state,residue_id,molecule_id,assembly_id
0,particle,BB-PDha,0,"[7.499999999999999, 7.499999999999999, 7.49999...",None,BB-PDha,0,0,<NA>
1,particle,COOH-PDha,1,"[7.770562720511088, 7.059202339683819, 6.77927...",None,COOH-PDha,0,0,<NA>
2,particle,NH3-PDha,2,"[7.749050564056614, 7.106238951223266, 6.85319...",None,NH3-PDha,0,0,<NA>
3,particle,BB-PDha,3,"[7.692460718229277, 6.8431412239990514, 7.9739...",None,BB-PDha,1,0,<NA>
4,particle,COOH-PDha,4,"[7.2107917459456745, 7.181071829992744, 8.6378...",None,COOH-PDha,1,0,<NA>
5,particle,NH3-PDha,5,"[7.847768489420128, 7.330068748851457, 8.58571...",None,NH3-PDha,1,0,<NA>


In [19]:
pmb.get_instances_df(pmb_type='bond')

,pmb_type,name,bond_id,particle_id1,particle_id2
0,bond,BB-PDha-COOH-PDha,0,0,1
1,bond,BB-PDha-NH3-PDha,1,0,2
2,bond,BB-PDha-COOH-PDha,2,3,4
3,bond,BB-PDha-NH3-PDha,3,3,5
4,bond,BB-PDha-BB-PDha,4,0,3


In [20]:
# Check residue instances
pmb.get_instances_df(pmb_type = 'residue')

,pmb_type,name,residue_id,molecule_id,assembly_id
0,residue,PDha_mon,0,0,<NA>
1,residue,PDha_mon,1,0,<NA>


In [21]:
# Check molecule instances
pmb.get_instances_df(pmb_type = 'molecule')

,pmb_type,name,molecule_id,assembly_id
0,molecule,PDha,0,<NA>


In [22]:
pmb.set_simulation_engine(espresso_system)
pmb.add_instances_to_engine()

Now, let us see what we have created...

In [23]:
picture_name = 'PDha_system.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                              filename = picture_name)
img = Image.open(picture_name)
img.show()

Delete the particles and check that there are no particle instances in the pyMBE database

In [25]:
pmb.delete_instances_in_system(instance_id=molecule_ids[0],
                               pmb_type="molecule", 
                              espresso_system = espresso_system)
# Check particle instances
pmb.get_instances_df(pmb_type = 'particle')

""


In [ ]:
particle_id_map = pmb.get_particle_id_map(object_name=cation_name)
# This will delete all particles that we created before
for pid in particle_id_map["all"]:
    pmb.delete_instances_in_system(instance_id=pid, 
                                   pmb_type="particle",
                                  espresso_system = espresso_system)

## How to create complex polymers <a class="anchor" id="complex_polymers"></a>

pyMBE can also be used to setup models that requiere more complex side chains, i.e. with more than one bead per side chain. One example of these complex molecules is the poly(N,N-diallylglutamate) (PDAGA), whose structure is depicted in the figure below. Following the logic of the previous example, one would construct PDAGA with pyMBE by defining a `residue` with a `central_bead` for the polymer backbone (grey) and a `side_chain` attached to it. In this case, the group in the side chain of the PDAGA monomer has a complex structure. This group can be coarse-grained by defining another `residue`  composed by a new `central_bead` which represents the cyclic amine group (blue) and two `side_chains` ($\alpha$ and $\beta$ carboxyl) attached to it (red and orange).

<img src="../figs/PDAGA.png" width=150 height=150 />

One can start by defining templates for each different bead of the PDAGA.

In [24]:
PDAGA_backbone_bead = 'BB-PDAGA'
PDAGA_cyclic_amine_bead = 'NH3-PDAGA'
PDAGA_alpha_carboxyl_bead = 'aCOOH-PDAGA'
PDAGA_beta_carboxyl_bead = 'bCOOH-PDAGA'

pmb.define_particle(name = PDAGA_backbone_bead, 
                    z = 0,
                    sigma = 0.4*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDAGA_cyclic_amine_bead, 
                    z = 0, 
                    sigma = 0.3*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDAGA_alpha_carboxyl_bead, 
                    z = 0, 
                    sigma = 0.2*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDAGA_beta_carboxyl_bead, 
                    z = 0, 
                    sigma = 0.4*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

The next step is to define the two different residue templates: 
1. The side chain: two carboxyl beads attached to the cyclic amine bead.

In [26]:
PDAGA_side_chain_residue = 'PDAGA_side_chain_residue'

pmb.define_residue (name = PDAGA_side_chain_residue,
                    central_bead = PDAGA_cyclic_amine_bead,
                    side_chains = [PDAGA_alpha_carboxyl_bead, PDAGA_beta_carboxyl_bead])

2. Each monomeric unit of the PDAGA: the side chain defined above attached to the backbone.

In [27]:
PDAGA_monomer_residue = 'PDAGA_monomer_residue'
pmb.define_residue( name = PDAGA_monomer_residue,
                    central_bead = PDAGA_backbone_bead,
                    side_chains = [PDAGA_side_chain_residue])

Then, we need to define bond templates in a similar way as for the case of the simple polymer.

In [28]:
bond_type = 'harmonic'
generic_bond_lenght=0.4 * pmb.units.nm
generic_harmonic_constant = 400 * pmb.units('reduced_energy / reduced_length**2')

harmonic_bond = {'r_0'    : generic_bond_lenght,
                 'k'      : generic_harmonic_constant,
                 }

pmb.define_bond(bond_type = bond_type,
                bond_parameters = harmonic_bond,
                particle_pairs = [[PDAGA_backbone_bead, PDAGA_backbone_bead],
                                  [PDAGA_backbone_bead, PDAGA_cyclic_amine_bead],
                                  [PDAGA_alpha_carboxyl_bead, PDAGA_cyclic_amine_bead],
                                  [PDAGA_beta_carboxyl_bead, PDAGA_cyclic_amine_bead]])

Now, let us define an octamer of PDAGA.

In [29]:
PDAGA_polymer = 'PDAGA'
N_monomers = 4

pmb.define_molecule(name = PDAGA_polymer,
                    residue_list = [PDAGA_monomer_residue]*N_monomers)

Finally, we are able to create a PDAGA polymer into the ESPResSo system.

In [39]:
N_polymers = 1

mol_ids = pmb.create_molecule(name = PDAGA_polymer,
                            number_of_molecules= N_polymers,
                            box_l= box_l,
                            list_of_first_residue_positions = [[Box_L.to('reduced_length').magnitude/2]*3])

In [40]:
pmb.get_instances_df(pmb_type = 'particle')

,pmb_type,name,particle_id,position,fix,initial_state,residue_id,molecule_id,assembly_id
0,particle,BB-PDAGA,0,"[7.499999999999999, 7.499999999999999, 7.49999...",None,BB-PDAGA,0,0,<NA>
1,particle,NH3-PDAGA,1,"[8.006035046780134, 6.983156096845676, 7.83500...",None,NH3-PDAGA,0,0,<NA>
2,particle,aCOOH-PDAGA,2,"[7.228892589749606, 6.828507543870596, 7.91016...",None,aCOOH-PDAGA,0,0,<NA>
3,particle,bCOOH-PDAGA,3,"[8.390296523191001, 7.449356309621872, 8.35503...",None,bCOOH-PDAGA,0,0,<NA>
4,particle,BB-PDAGA,4,"[7.033629851553653, 6.866558505888692, 7.22719...",None,BB-PDAGA,1,0,<NA>
5,particle,NH3-PDAGA,5,"[6.874476454747638, 6.659394433454941, 7.98031...",None,NH3-PDAGA,1,0,<NA>
6,particle,aCOOH-PDAGA,6,"[6.785798310596719, 6.068056544244533, 7.45498...",None,aCOOH-PDAGA,1,0,<NA>
7,particle,bCOOH-PDAGA,7,"[6.179699453438731, 7.035070100567888, 8.08793...",None,bCOOH-PDAGA,1,0,<NA>
8,particle,BB-PDAGA,8,"[6.567259703107307, 6.233117011777386, 6.95439...",None,BB-PDAGA,2,0,<NA>
9,particle,NH3-PDAGA,9,"[7.17699479917509, 5.733385942628339, 7.072389...",None,NH3-PDAGA,2,0,<NA>


In [41]:
pmb.get_instances_df(pmb_type = 'residue')

,pmb_type,name,residue_id,molecule_id,assembly_id
0,residue,PDAGA_monomer_residue,0,0,<NA>
1,residue,PDAGA_monomer_residue,1,0,<NA>
2,residue,PDAGA_monomer_residue,2,0,<NA>
3,residue,PDAGA_monomer_residue,3,0,<NA>


In [42]:
pmb.get_instances_df(pmb_type = 'bond')

,pmb_type,name,bond_id,particle_id1,particle_id2
0,bond,NH3-PDAGA-aCOOH-PDAGA,0,1,2
1,bond,NH3-PDAGA-bCOOH-PDAGA,1,1,3
2,bond,BB-PDAGA-NH3-PDAGA,2,0,1
3,bond,NH3-PDAGA-aCOOH-PDAGA,3,5,6
4,bond,NH3-PDAGA-bCOOH-PDAGA,4,5,7
5,bond,BB-PDAGA-NH3-PDAGA,5,4,5
6,bond,BB-PDAGA-BB-PDAGA,6,0,4
7,bond,NH3-PDAGA-aCOOH-PDAGA,7,9,10
8,bond,NH3-PDAGA-bCOOH-PDAGA,8,9,11
9,bond,BB-PDAGA-NH3-PDAGA,9,8,9


In [43]:
pmb.set_simulation_engine(espresso_system)
pmb.add_instances_to_engine()

Now, let us see our PDAGA molecule.

In [44]:
picture_name = 'PDAGA_system.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                               filename = picture_name)
img = Image.open(picture_name)
img.show()


Delete the particles and check that the pyMBE database is empty.

In [37]:

for mol_id in mol_ids:
    pmb.delete_instances_in_system(instance_id=mol_id,
                                   pmb_type="molecule",
                                   espresso_system=espresso_system)
pmb.get_instances_df(pmb_type = 'particle')

""


## How to create di-block copolymers <a class="anchor" id="diblock_copolymers"></a>

In turn, the residues previously defined to build the PDAGA and PDha molecules can be used to build more complex polymers such as a di-block PDha-PDAGA copolymer, as shown in the picture below

<img src="../figs/PDAGA_PDha_diblock_copolymer.png" width=250 height=250 />

Define the bond template between the backbone particle of PDha and the backbone particle of PDAGA

In [48]:

pmb.define_bond(bond_type = bond_type,
                bond_parameters = harmonic_bond,
                particle_pairs = [[PDha_backbone_bead, PDAGA_backbone_bead]])

Define a molecule template for the di-block polymer molecule using Python list comprehension methods

In [49]:
N_monomers_PDha = 4
N_monomers_PDAGA = 4
diblock_polymer = 'diblock'

pmb.define_molecule(name = diblock_polymer,
                    residue_list = [PDha_residue]*N_monomers_PDha+[PDAGA_monomer_residue]*N_monomers_PDAGA)

Creating the di-block polymer into the ESPResSo system

In [50]:
N_polymers = 1

mol_ids = pmb.create_molecule(name = diblock_polymer,
                              number_of_molecules= N_polymers,
                              box_l= box_l,
                              list_of_first_residue_positions = [[Box_L.to('reduced_length').magnitude/2]*3]) 
# See the particle instances you have created
pmb.get_instances_df(pmb_type="particle")

,pmb_type,name,particle_id,position,fix,initial_state,residue_id,molecule_id,assembly_id
0,particle,BB-PDha,0,"[7.499999999999999, 7.499999999999999, 7.49999...",None,BB-PDha,0,0,<NA>
1,particle,COOH-PDha,1,"[7.571167527771879, 7.604628201186956, 8.37803...",None,COOH-PDha,0,0,<NA>
2,particle,NH3-PDha,2,"[7.61187816904253, 7.692330966595771, 6.734544...",None,NH3-PDha,0,0,<NA>
3,particle,BB-PDha,3,"[8.209680803443417, 7.064694517707376, 7.49434...",None,BB-PDha,1,0,<NA>
4,particle,COOH-PDha,4,"[7.96317931944196, 6.672642643457426, 6.737725...",None,COOH-PDha,1,0,<NA>
5,particle,NH3-PDha,5,"[7.949794989096124, 6.632984135848388, 8.11201...",None,NH3-PDha,1,0,<NA>
6,particle,BB-PDha,6,"[8.919361606886834, 6.629389035414753, 7.48869...",None,BB-PDha,2,0,<NA>
7,particle,COOH-PDha,7,"[8.542953602465508, 6.0225607419657345, 6.9623...",None,COOH-PDha,2,0,<NA>
8,particle,NH3-PDha,8,"[9.250938657970849, 7.163600709092095, 7.97870...",None,NH3-PDha,2,0,<NA>
9,particle,BB-PDha,9,"[9.629042410330252, 6.194083553122129, 7.48304...",None,BB-PDha,3,0,<NA>


In [52]:
pmb.add_instances_to_engine()

Now, let us see our di-block PDha-PDAGA molecule.

In [53]:
picture_name = 'diblock_system.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                               filename = picture_name)
img = Image.open(picture_name)
img.show()

Delete the particles and check that our df is empty.

In [54]:
for mol_id in mol_ids:
    pmb.delete_instances_in_system(instance_id=0, 
                                   pmb_type="molecule",
                                   espresso_system = espresso_system)
pmb.get_instances_df(pmb_type="particle")

""


## Practice by creating a custom polyampholyte chain <a class="anchor" id="branched_polyampholytes"></a>

Polyampholytes are polymers containing both acidic and basic groups on the same molecule, one example of a branched polyampholyte is depicted in the figure below.

<img src="../figs/branched_polyampholyte.png" width=350 height=350 />

We will create the polyampholyte chain in the figure, starting by defining two different residues, 'Res_1' and 'Res_2'. The polyampholyte chain is then defined by following residue_list:

residue_list = 2*["Res_1"] + ["Res_2"] + 2*["Res_1"] + 2*["Res_2"]

### Tasks to do:

1. Define particle templates for each different bead in the residues using "pmb.define_particle". There are 3 different particles, an inert particle, an acidic particle with pKa = 4, and a basic particle with pKa = 9.
2. Define residue templates using "pmb.define_residue". "Res_1" consists of an inert particle as central bead and acidic and basic particles as side chain. "Res_2" consists of an inert particle as central bead and "Res_1" as side chain.
3. Define bond templates for each pair of particle templates.  
4. Define a molecule template for the branched polyampholyte chain using "pmb.define_molecule" with the above "residue_list." 
5. Create the branched polyampholyte into the ESPResSo system.
6. Visualize your creation.
7. Delete the molecule and check that the pyMBE database is empty.

#### 1. Define particle templates for each different bead of Res_1 and Res_2.

In [55]:
PDha_backbone_bead = 'BB-part1'
PDha_carboxyl_bead = 'COOH-part1'
PDha_amine_bead = 'NH3-part1'

pmb.define_particle(name = PDha_backbone_bead, 
                    z = 0, 
                    sigma = 0.4*pmb.units.nm,
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDha_carboxyl_bead, 
                    z = 0, 
                    sigma = 0.5*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))

pmb.define_particle(name = PDha_amine_bead, 
                    z = 0, 
                    sigma = 0.3*pmb.units.nm, 
                    epsilon = 1*pmb.units('reduced_energy'))



#### 2. Define the residue templates for Res_1 and Res_2.

In [47]:
pmb.get_templates_df('particle')
# pmb.get_instances_df('particle')

,pmb_type,name,sigma,epsilon,cutoff,offset,initial_state
0,particle,Na,0.35 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,Na
1,particle,BB-PDha,0.4 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,BB-PDha
2,particle,COOH-PDha,0.5 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,COOH-PDha
3,particle,NH3-PDha,0.3 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,NH3-PDha
4,particle,BB-PDAGA,0.4 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,BB-PDAGA
5,particle,NH3-PDAGA,0.3 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,NH3-PDAGA
6,particle,aCOOH-PDAGA,0.2 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,aCOOH-PDAGA
7,particle,bCOOH-PDAGA,0.4 nanometer,25.692579121085853 millielectron_volt,0.5612310241546864 nanometer,0.0 nanometer,bCOOH-PDAGA


In [57]:
pmb.get_templates_df('particle_state')

,pmb_type,name,particle_name,z,es_type
0,particle_state,Na,Na,0,0
1,particle_state,BB-PDha,BB-PDha,0,1
2,particle_state,COOH-PDha,COOH-PDha,0,2
3,particle_state,NH3-PDha,NH3-PDha,0,3
4,particle_state,BB-PDAGA,BB-PDAGA,0,4
5,particle_state,NH3-PDAGA,NH3-PDAGA,0,5
6,particle_state,aCOOH-PDAGA,aCOOH-PDAGA,0,6
7,particle_state,bCOOH-PDAGA,bCOOH-PDAGA,0,7
8,particle_state,BB-part1,BB-part1,0,8
9,particle_state,COOH-part1,COOH-part1,0,9


In [59]:
pmb.get_templates_df('residue')

,pmb_type,name,central_bead,side_chains
0,residue,PDha_mon,BB-PDha,"[COOH-PDha, NH3-PDha]"
1,residue,PDAGA_side_chain_residue,NH3-PDAGA,"[aCOOH-PDAGA, bCOOH-PDAGA]"
2,residue,PDAGA_monomer_residue,BB-PDAGA,[PDAGA_side_chain_residue]


In [60]:
pmb.get_templates_df('molecule')

,pmb_type,name,residue_list
0,molecule,PDha,"[PDha_mon, PDha_mon, PDha_mon, PDha_mon, PDha_..."
1,molecule,PDAGA,"[PDAGA_monomer_residue, PDAGA_monomer_residue,..."
2,molecule,diblock,"[PDha_mon, PDha_mon, PDha_mon, PDha_mon, PDAGA..."


In [61]:
pmb.db.delete_templates("bond")

In [62]:
pmb.db.delete_instances('particle_state')

In [63]:
res1='Residue1'
pmb.define_residue(name = res1,
                    central_bead = PDha_backbone_bead,
                    side_chains = [PDha_carboxyl_bead,PDha_amine_bead])

side_chain_name_res2='side_chain_res_2'
pmb.define_residue( name = side_chain_name_res2,
                    central_bead = PDha_backbone_bead,
                    side_chains = [PDha_carboxyl_bead,PDha_amine_bead])

res2='Residue2'
pmb.define_residue(name = res2,
                    central_bead = PDha_backbone_bead,
                    side_chains = [side_chain_name_res2])

3. Define bond templates for each pair of particle templates.  

In [64]:
print(dir(pmb))
print(pmb.get_instances_df(pmb_type="residue"))
pmb.get_instances_df(pmb_type = 'bond')

['Kw', 'N_A', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_check_bond_inputs', '_check_dimensionality', '_check_pka_set', '_create_espresso_bond_instance', '_create_hydrogel_chain', '_create_hydrogel_node', '_delete_particles_from_espresso', '_get_espresso_bond_instance', '_get_label_id_map', '_get_residue_list_from_sequence', '_get_template_type', 'add_instances_to_engine', 'calculate_HH', 'calculate_HH_Donnan', 'calculate_center_of_mass', 'calculate_net_charge', 'center_object_in_simulation_box', 'create_added_salt', 'create_bond', 'create_counterions', 'create_hydrogel', 'create_molecule', 'create_particle', 'create_protein', 'create_residue', 'db', 'define_bond', 'define_default_bond', 

""


In [65]:
bond_type = 'harmonic'
generic_bond_lenght=0.4 * pmb.units.nm
generic_harmonic_constant = 400 * pmb.units('reduced_energy / reduced_length**2')

harmonic_bond = {'r_0'    : generic_bond_lenght,
                 'k'      : generic_harmonic_constant,
                 }

pmb.define_bond(bond_type = bond_type,
                bond_parameters = harmonic_bond,
                particle_pairs = [[PDha_backbone_bead, PDha_backbone_bead],
                                  [PDha_backbone_bead, PDha_carboxyl_bead],
                                  [PDha_backbone_bead, PDha_amine_bead]])

#### 3. Define a molecule template for the diblock polyampholyte. 

In [66]:
diblock_polymer='custom_polyampholyte'
pmb.define_molecule(name = diblock_polymer,
                    residue_list = 2*[res1]+[res2]+2*[res1]+2*[res2])

#### 4. Create the diblock polyampholyte chain into the ESPResSo system.

In [68]:
N_polymers = 1

mol_ids = pmb.create_molecule(name = diblock_polymer,
                              number_of_molecules= N_polymers,
                              box_l = box_l,
                              list_of_first_residue_positions = [[Box_L.to('reduced_length').magnitude/2]*3]) 
# See the particle instances you have created
pmb.get_instances_df(pmb_type="particle")

,pmb_type,name,particle_id,position,fix,initial_state,residue_id,molecule_id,assembly_id
0,particle,BB-part1,0,"[7.499999999999999, 7.499999999999999, 7.49999...",None,BB-part1,0,0,<NA>
1,particle,COOH-part1,1,"[7.523286897123127, 6.7825727593903675, 8.0212...",None,COOH-part1,0,0,<NA>
2,particle,NH3-part1,2,"[7.37125134474576, 7.37461010141046, 8.2766151...",None,NH3-part1,0,0,<NA>
3,particle,BB-part1,3,"[6.694202325055984, 7.360411350475328, 7.34387...",None,BB-part1,1,0,<NA>
4,particle,COOH-part1,4,"[6.8599927657093085, 7.377868784266711, 6.4725...",None,COOH-part1,1,0,<NA>
5,particle,NH3-part1,5,"[6.5574917981405205, 7.276291078140914, 8.1246...",None,NH3-part1,1,0,<NA>
6,particle,BB-part1,6,"[5.888404650111968, 7.220822700950657, 7.18775...",None,BB-part1,2,0,<NA>
7,particle,BB-part1,7,"[6.0263924170157, 7.333839572897158, 6.3745136...",None,BB-part1,2,0,<NA>
8,particle,COOH-part1,8,"[5.428627933302976, 7.973911138422894, 6.51570...",None,COOH-part1,2,0,<NA>
9,particle,NH3-part1,9,"[6.214629818479694, 6.6498848824883945, 6.7381...",None,NH3-part1,2,0,<NA>


In [69]:
pmb.get_instances_df(pmb_type="residue")

,pmb_type,name,residue_id,molecule_id,assembly_id
0,residue,Residue1,0,0,<NA>
1,residue,Residue1,1,0,<NA>
2,residue,Residue2,2,0,<NA>
3,residue,Residue1,3,0,<NA>
4,residue,Residue1,4,0,<NA>
5,residue,Residue2,5,0,<NA>
6,residue,Residue2,6,0,<NA>


In [70]:
pmb.add_instances_to_engine()

#### 5. Visualize your creation.

In [71]:
picture_name = 'CustomPolyampholite2_new.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                               filename = picture_name)
img = Image.open(picture_name)
img.show()


#### 6. Delete the molecule and check that the pyMBE database is empty.

In [72]:
for mol_id in mol_ids:
    pmb.delete_instances_in_system(instance_id=mol_id,
                                   pmb_type="molecule",
                                   espresso_system=espresso_system)
pmb.get_instances_df(pmb_type = 'particle')

""


Refer to the sample script "branched_polyampholyte.py" in the samples folder for a complete solution of this exercise.

## How to create peptides <a class="anchor" id="peptides"></a>

pyMBE includes built-on functions to facilitate the setting up of coarse-grained models for peptides from their aminoacid sequence. Currently, there are two different coarse-grained models implemented: 

* `1beadAA`, where the aminoacid is represented by one single bead.
* `2beadAA`, where the aminoacid is represented by two beads (backbone and side-chain). 

We provide reference parameters in the folder (`pyMBE/parameters`) which can be loaded into pyMBE. The peptide sequence should be provided as a `str` composed either by the list of the one letter code or the list of the three letter code of the corresponding aminoacids. For example, the two possible ways to provide the peptide Cysteine$_3$ - Glutamic acid$_2$ - Histidine$_4$ - Valine are:

* one letter code: 'CCCEEHHHHV'
* three letter code: 'CYS-CYS-CYS-GLU-GLU-HIS-HIS-HIS-HIS-VAL'

Let's set up the peptide Lysine$_5$ - Glutamic acid$_5$ using a two beads coarse-grained model.

In [73]:
N_peptide = 1
sequence = "KKKKKEEEEE"
model = '2beadAA'

We can use the peptide parametrization reported by Lunkad et al. [2], which is provided in the reference folder. This parametrization includes information about the particles (i.e. their Lennard-Jones parameters) and their bonding potentials (harmonic bonds).

In [74]:
path_to_interactions=pmb.root / "parameters" / "peptides" / "Lunkad2021"
path_to_pka=pmb.root / "parameters" / "pka_sets" / "Hass2015.json"
pmb.load_database(folder=path_to_interactions) 
pmb.get_templates_df(pmb_type="particle")
pmb.get_templates_df(pmb_type="bond")

,pmb_type,name,bond_type,particle_name1,particle_name2,parameters
0,bond,CA-CA,harmonic,CA,CA,"{'r_0': 0.382 nanometer, 'k': 10277.0316484343..."
1,bond,CA-D,harmonic,CA,D,"{'r_0': 0.329 nanometer, 'k': 10277.0316484343..."
2,bond,CA-E,harmonic,CA,E,"{'r_0': 0.435 nanometer, 'k': 10277.0316484343..."
3,bond,CA-H,harmonic,CA,H,"{'r_0': 0.452 nanometer, 'k': 10277.0316484343..."
4,bond,CA-Y,harmonic,CA,Y,"{'r_0': 0.648 nanometer, 'k': 10277.0316484343..."
5,bond,CA-K,harmonic,CA,K,"{'r_0': 0.558 nanometer, 'k': 10277.0316484343..."


Additionally, we can load one of the reference sets of pKa values for amino acids that we provide in pyMBE

In [75]:
pmb.load_pka_set(path_to_pka)
# Check the loaded pKa set
pmb.get_reactions_df()

,reaction,stoichiometry,pK,reaction_type,metadata,simulation_method
0,DH <-> D,"{'DH': -1, 'D': 1}",4.0,monoprotic_acid,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
1,EH <-> E,"{'EH': -1, 'E': 1}",4.4,monoprotic_acid,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
2,YH <-> Y,"{'YH': -1, 'Y': 1}",9.6,monoprotic_acid,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
3,CH <-> C,"{'CH': -1, 'C': 1}",8.3,monoprotic_acid,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
4,HH <-> H,"{'HH': -1, 'H': 1}",6.8,monoprotic_base,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
5,KH <-> K,"{'KH': -1, 'K': 1}",10.4,monoprotic_base,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
6,RH <-> R,"{'RH': -1, 'R': 1}",13.5,monoprotic_base,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
7,nH <-> n,"{'nH': -1, 'n': 1}",8.0,monoprotic_base,"{'summary': 'pKa-values of Hass et al.', 'sour...",None
8,cH <-> c,"{'cH': -1, 'c': 1}",3.6,monoprotic_acid,"{'summary': 'pKa-values of Hass et al.', 'sour...",None


Since monoprotic acid/base particles can be in two possible states, protonated and deprotonated, we need to define templates for those particle states

In [76]:
# Get the pKa set stored in pyMBE
pka_set = pmb.get_pka_set()
# Check the pka_set
print(pka_set)
# define templates for the different particle states of monoprotic acid an basic groups:
for acidbase_particle in pka_set.keys():
    pmb.define_monoprototic_particle_states(particle_name=acidbase_particle,
                                            acidity=pka_set[acidbase_particle]["acidity"])
pmb.get_templates_df(pmb_type="particle_state")

{'D': {'pka_value': 4.0, 'acidity': 'acidic'}, 'E': {'pka_value': 4.4, 'acidity': 'acidic'}, 'Y': {'pka_value': 9.6, 'acidity': 'acidic'}, 'C': {'pka_value': 8.3, 'acidity': 'acidic'}, 'H': {'pka_value': 6.8, 'acidity': 'basic'}, 'K': {'pka_value': 10.4, 'acidity': 'basic'}, 'R': {'pka_value': 13.5, 'acidity': 'basic'}, 'n': {'pka_value': 8.0, 'acidity': 'basic'}, 'c': {'pka_value': 3.6, 'acidity': 'acidic'}}


,pmb_type,name,particle_name,z,es_type
0,particle_state,CA,CA,0,0
1,particle_state,DH,D,0,1
2,particle_state,D,D,-1,2
3,particle_state,EH,E,0,3
4,particle_state,E,E,-1,4
5,particle_state,YH,Y,0,5
6,particle_state,Y,Y,-1,6
7,particle_state,CH,C,0,7
8,particle_state,C,C,-1,8
9,particle_state,HH,H,1,9


In [77]:
pmb.get_templates_df(pmb_type="particle")

,pmb_type,name,sigma,epsilon,cutoff,offset,initial_state
0,particle,CA,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,CA
1,particle,D,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,DH
2,particle,E,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,EH
3,particle,H,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,HH
4,particle,Y,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,YH
5,particle,K,0.35 nanometer,25.69257912108585 millielectron_volt,0.3984740271498274 nanometer,0.0 nanometer,KH


The above functions define templates for particles and particle states Before creating a peptide molecule, we also need to define templates for the aminoacid residues and the peptide molecule

In [78]:
from pyMBE.lib.handy_functions import define_peptide_AA_residues

# This is a convinience function that defines residue templates
# for aminoacids based on some pre-defined models
define_peptide_AA_residues(sequence=sequence,
                           model=model,
                           pmb=pmb)
pmb.define_peptide(name = sequence, 
                   sequence = sequence, 
                   model = model)
pmb.get_templates_df(pmb_type="residue")
pmb.get_templates_df(pmb_type="peptide")

,pmb_type,name,model,residue_list,sequence
0,peptide,KKKKKEEEEE,2beadAA,"[AA-K, AA-K, AA-K, AA-K, AA-K, AA-E, AA-E, AA-...",KKKKKEEEEE


Now, we can create instances of our peptide template into the ESPResSo system. 

In [80]:

mol_ids = pmb.create_molecule(name = sequence,
                            number_of_molecules=  N_peptide,
                            box_l = box_l,
                            list_of_first_residue_positions = [[Box_L.to('reduced_length').magnitude/2]*3])
pmb.get_instances_df(pmb_type="peptide")

,pmb_type,name,molecule_id,assembly_id
0,peptide,KKKKKEEEEE,0,<NA>


In [82]:
pmb.add_instances_to_engine()

Let us visualize our peptide.

In [83]:
picture_name = 'peptide_new.png'
create_snapshot_of_espresso_system(espresso_system = espresso_system, 
                               filename = picture_name)
img = Image.open(picture_name)
img.show()

Delete the particles and check that the pyMBE database is empty

In [84]:
for mol_id in mol_ids:
    pmb.delete_instances_in_system(instance_id=0,
                                   pmb_type="peptide", 
                                   espresso_system = espresso_system)

### References

[1] Beyer, D., Torres, P. B., Pineda, S. P., Narambuena, C. F., Grad, J. N., Košovan, P., & Blanco, P. M. (2024). pyMBE: The Python-based molecule builder for ESPResSo. The Journal of Chemical Physics, 161(2).
[2] Lunkad, R., Murmiliuk, A., Hebbeker, P., Boublík, M., Tošner, Z., Štěpánek, M., & Košovan, P. (2021). Quantitative prediction of charge regulation in oligopeptides. Molecular Systems Design & Engineering, 6(2), 122-131.